<a href="https://colab.research.google.com/github/Pranjli-S/Assignment_T20-EDA-/blob/main/T20_EDA_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import ast

# 1. Load Data & Explore Sanity
df = pd.read_csv('International_T20_Data.csv')
print("--- Dataset Sanity Check ---")
print(df.info())
print("\nFirst 5 rows:")
display(df.head())

# 2. Rename Columns
# Replacing "info.", "meta.", and other unnecessary prefixes for cleaner names
col_mapping = {}
for col in df.columns:
    new_col = col.replace('info.outcome.by.', 'win_by_')\
                 .replace('info.outcome.', '')\
                 .replace('info.toss.', 'toss_')\
                 .replace('info.', '')\
                 .replace('meta.created', 'created_date')\
                 .replace('meta.', '')\
                 .replace('.', '_')
    col_mapping[col] = new_col

df.rename(columns=col_mapping, inplace=True)
print("\n--- Renamed Columns ---")
print(df.columns.tolist())

# 3. Top three venues which hosted the greatest number of matches
print("\n--- Top 3 Venues by Matches Hosted ---")
top_venues = df['venue'].value_counts().head(3)
print(top_venues)

# 4. Pair of cricket teams who played the most T20 matches against each other
print("\n--- Most Frequent Matchup (Team Pair) ---")
# Using ast.literal_eval to safely evaluate string lists, then sorting to match pairs properly
df['teams_list'] = df['teams'].apply(ast.literal_eval)
df['team_pair'] = df['teams_list'].apply(lambda x: tuple(sorted(x)))
top_pair = df['team_pair'].value_counts().head(1)
print(top_pair)

# 5. Top five teams by win percentages (Matches won / Matches played * 100)
print("\n--- Top 5 Teams by Win Percentage ---")
team_stats = {}
for idx, row in df.iterrows():
    teams = row['teams_list']
    winner = row['winner']
    for team in teams:
        if team not in team_stats:
            team_stats[team] = {'played': 0, 'won': 0}
        team_stats[team]['played'] += 1
        # If the match wasn't a tie or no-result, and this team won
        if pd.notna(winner) and team == winner:
            team_stats[team]['won'] += 1

stats_df = pd.DataFrame.from_dict(team_stats, orient='index')
stats_df['win_pct'] = (stats_df['won'] / stats_df['played']) * 100
top_5_teams = stats_df.sort_values(by='win_pct', ascending=False).head(5)
display(top_5_teams)

# 6. Function to get the scorecard of each match
def get_scorecard(innings_str):
    innings = ast.literal_eval(innings_str)

    def process_inning(inning_name):
        inning_dict = next((i[inning_name] for i in innings if inning_name in i), None)
        if not inning_dict:
            return pd.DataFrame()

        deliveries = inning_dict['deliveries']
        batsmen = {}
        bowlers = {}

        for d in deliveries:
            val = list(d.values())[0]
            batsman, bowler = val['batsman'], val['bowler']

            # Calculate Batsman Runs
            batsmen[batsman] = batsmen.get(batsman, 0) + val['runs']['batsman']

            # Calculate Bowler Stats
            if bowler not in bowlers:
                bowlers[bowler] = {'Wickets': 0, 'Runs Conceded': 0}
            bowlers[bowler]['Runs Conceded'] += val['runs']['total'] # total runs off bowler

            # Wickets calculation (excluding run outs, retired hurt, etc)
            if 'wicket' in val and val['wicket'].get('kind') not in ['run out', 'retired hurt', 'obstructing the field', 'retired not out']:
                bowlers[bowler]['Wickets'] += 1

        # Get Top 4 batsmen
        df_bat = pd.DataFrame(list(batsmen.items()), columns=['Batsman', 'Runs Scored'])
        df_bat = df_bat.sort_values('Runs Scored', ascending=False).head(4).reset_index(drop=True)

        # Get Top 4 bowlers (Sorted by Wickets descending, then Runs Conceded ascending)
        df_bowl = pd.DataFrame([{'Bowler': b, 'Wickets': stats['Wickets'], 'Runs Conceded': stats['Runs Conceded']} for b, stats in bowlers.items()])
        df_bowl = df_bowl.sort_values(['Wickets', 'Runs Conceded'], ascending=[False, True]).head(4).reset_index(drop=True)

        # Merge both into one Dataframe
        return pd.concat([df_bat, df_bowl], axis=1)

    df1 = process_inning('1st innings')
    df2 = process_inning('2nd innings')
    return df1, df2

print("\n--- Sample Scorecard for the 1st Match ---")
df_first, df_second = get_scorecard(df['innings'][0])

print("Team 1 Batsmen vs Team 2 Bowlers:")
display(df_first)

print("\nTeam 2 Batsmen vs Team 1 Bowlers:")
display(df_second)

--- Dataset Sanity Check ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1417 entries, 0 to 1416
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   innings                      1417 non-null   object 
 1   meta.data_version            1417 non-null   float64
 2   meta.created                 1417 non-null   object 
 3   meta.revision                1417 non-null   int64  
 4   info.dates                   1417 non-null   object 
 5   info.gender                  1417 non-null   object 
 6   info.match_type              1417 non-null   object 
 7   info.outcome.by.wickets      651 non-null    float64
 8   info.outcome.winner          1372 non-null   object 
 9   info.overs                   1417 non-null   int64  
 10  info.player_of_match         1255 non-null   object 
 11  info.teams                   1417 non-null   object 
 12  info.toss.decision           1417 non-null   ob

,innings,meta.data_version,meta.created,meta.revision,info.dates,info.gender,info.match_type,info.outcome.by.wickets,info.outcome.winner,info.overs,...,info.outcome.by.runs,info.match_type_number,info.neutral_venue,info.outcome.method,info.outcome.result,info.outcome.eliminator,info.supersubs.New Zealand,info.supersubs.South Africa,info.bowl_out,info.outcome.bowl_out
0,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-18,2,"[datetime.date(2017, 2, 17)]",male,T20,5.0,Sri Lanka,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-19,2,"[datetime.date(2017, 2, 19)]",male,T20,2.0,Sri Lanka,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-23,1,"[datetime.date(2017, 2, 22)]",male,T20,NaN,Australia,20,...,41.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[{'1st innings': {'team': 'Hong Kong', 'delive...",0.9,2016-09-12,1,"[datetime.date(2016, 9, 5)]",male,T20,NaN,Hong Kong,20,...,40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"[{'1st innings': {'team': 'Zimbabwe', 'deliver...",0.9,2016-06-19,1,"[datetime.date(2016, 6, 18)]",male,T20,NaN,Zimbabwe,20,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- Renamed Columns ---
['innings', 'data_version', 'created_date', 'revision', 'dates', 'gender', 'match_type', 'win_by_wickets', 'winner', 'overs', 'player_of_match', 'teams', 'toss_decision', 'toss_winner', 'umpires', 'venue', 'city', 'win_by_runs', 'match_type_number', 'neutral_venue', 'method', 'result', 'eliminator', 'supersubs_New Zealand', 'supersubs_South Africa', 'bowl_out', 'bowl_out']

--- Top 3 Venues by Matches Hosted ---
venue
Dubai International Cricket Stadium    62
Sheikh Zayed Stadium                   41
Shere Bangla National Stadium          39
Name: count, dtype: int64

--- Most Frequent Matchup (Team Pair) ---
team_pair
(Australia, England)    45
Name: count, dtype: int64

--- Top 5 Teams by Win Percentage ---


,played,won,win_pct
Belgium,3,3,100.000000
Spain,6,5,83.333333
Germany,17,13,76.470588
Namibia,34,25,73.529412
Afghanistan,75,51,68.000000



--- Sample Scorecard for the 1st Match ---
Team 1 Batsmen vs Team 2 Bowlers:


,Batsman,Runs Scored,Bowler,Wickets,Runs Conceded
0,AJ Finch,43,SL Malinga,2,29
1,M Klinger,38,DAS Gunaratne,1,11
2,TM Head,31,PADLR Sandakan,1,31
3,AJ Turner,18,JRMVB Sanjaya,1,35



Team 2 Batsmen vs Team 1 Bowlers:


,Batsman,Runs Scored,Bowler,Wickets,Runs Conceded
0,DAS Gunaratne,52,AJ Turner,2,12
1,EMDY Munaweera,44,A Zampa,2,26
2,N Dickwella,30,PJ Cummins,1,30
3,TAM Siriwardana,15,JP Faulkner,0,29
